Author: Krish

In [1]:
# Importing relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import warnings
warnings.filterwarnings("ignore")

### User input required
Put the data path on your system in the cell below

Note: Assumes data has been restricted only to datasets that included the 'Termination Reason' column 

In [2]:
data_path = "/Users/viviadams/Downloads/CAR_Includes_Termination"

### User input ends

### Reading all filenames in the data folder

In [3]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))
df_main = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 'Activity Start Timestamp', 
                                'Queue Name', 'Agent Name', 'Termination Reason'])
df_main

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason


### Reading all data files
The code chunk below reads and appends all the CAR data files. The first two rows of each file are blank and thus ignored.

In [4]:
i=0
for f in files:
    i = i + 1
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
   # print(i, f.stem, df.shape) - removed printing file name

### Time datatype conversion
The code chunk below converts time from string to datetime datatype.

In [5]:
df_main["Activity Start Timestamp"] = df_main["Activity Start Timestamp"].apply(
    lambda x: datetime.strptime(x, "%Y/%m/%d %I:%M:%S %p"))

In [6]:
# Checking the datatype of all columns
df_main.dtypes

Contact Session ID                  object
EP Name                             object
Flow Name                           object
Activity Name                       object
Activity Start Timestamp    datetime64[ns]
Queue Name                          object
Agent Name                          object
Termination Reason                  object
dtype: object

In [7]:
# Creating a new column 'hour' as it will be useful to visualize peak calling hours
df_main["hour"] = df_main["Activity Start Timestamp"].dt.hour

## Data Cleaning & Analysis 

In [8]:
# Getting Contact Session IDs for calls where customer left 
customer_left_calls = df_main.loc[df_main['Termination Reason'] == 'Customer Left', 'Contact Session ID']

# Filtering DF to only include those Contact Session IDs 
terminated_by_customer = df_main.loc[df_main['Contact Session ID'].isin(customer_left_calls), :]


first_last_times = terminated_by_customer.groupby("Contact Session ID")["Activity Start Timestamp"].agg(["first", "last"])
first_last_times["Duration"] = first_last_times["last"] - first_last_times["first"]

terminated_by_customer = terminated_by_customer.merge(first_last_times["Duration"], on="Contact Session ID", how="left")

terminated_by_customer.head()

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour,Duration
0,ced63a4a-89f4-4e11-94b3-447a01cd34cb,Main Number Telephony EP,NaN,NaN,2025-03-16 00:40:10,NaN,NaN,NaN,0,0 days 00:00:02
1,ced63a4a-89f4-4e11-94b3-447a01cd34cb,NaN,LACMain,NaN,2025-03-16 00:40:10,NaN,NaN,NaN,0,0 days 00:00:02
2,ced63a4a-89f4-4e11-94b3-447a01cd34cb,Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-03-16 00:40:10,NaN,NaN,NaN,0,0 days 00:00:02
3,ced63a4a-89f4-4e11-94b3-447a01cd34cb,Main Number Telephony EP,LACMain,NaN,2025-03-16 00:40:10,NaN,NaN,NaN,0,0 days 00:00:02
4,ced63a4a-89f4-4e11-94b3-447a01cd34cb,Main Number Telephony EP,NaN,NaN,2025-03-16 00:40:12,NaN,NaN,Customer Left,0,0 days 00:00:02


In [9]:
# Utilizing Meera's data cleaning 

calls_customer_left = terminated_by_customer.copy() 

#Give each session ID an ID number in order of appearance
calls_customer_left["Call ID"] = calls_customer_left["Contact Session ID"].map(
    {id_: i+1 for i, id_ in enumerate(calls_customer_left["Contact Session ID"].unique())}
)

#Calculate Time Difference between rows 
calls_customer_left["Time Difference"] = (
    calls_customer_left.groupby("Contact Session ID")["Activity Start Timestamp"]
    .diff()
    .dt.total_seconds()  
)

calls_customer_left = calls_customer_left[
    [
        'Call ID',
        'Contact Session ID',
        'EP Name',
        'Flow Name',
        'Activity Name',
        'Queue Name',
        'Agent Name', 
        'Termination Reason',
        'Activity Start Timestamp',
        'Time Difference', # seconds
        'Duration'
    ]
]

calls_customer_left['Time Difference'] = calls_customer_left['Time Difference'].fillna(0)

### Identifying Calls Abandoned While Listening to Hold Music

In [10]:
# Finding Contact Session IDs where caller listened to hold music 
hold_music_calls = calls_customer_left.loc[calls_customer_left['Activity Name'] == 'PlayMOH300s', 'Contact Session ID']

# Filtering to only include those Contact Session IDs 
music_calls = calls_customer_left.loc[calls_customer_left['Contact Session ID'].isin(hold_music_calls), :]

In [11]:
# Sort by Call ID so all steps in a customer's call journey will be grouped together (i.e. no non-consecutive entries) & follow progression of their steps based on time 
music_calls.sort_values(by=['Call ID', 'Activity Start Timestamp'], inplace=True)

# Resetting index so that indices will correspond to their new order 
music_calls.reset_index(drop=True, inplace=True)

In [12]:
# Since values have been sorted & index reset, if the customer left midway through hold music, row where 'Activity Name' = 'PlayMOH300s' 
    # should be directly above the row with the termination reason 

# From this, built a function around one call as an example of logic for how to check if they left before music finished 
abandoned_during_MOH = 0 
index_of_interest = 28

if (music_calls.loc[(index_of_interest - 1), 'Activity Name'] == 'PlayMOH300s') and (music_calls.loc[index_of_interest, 'Time Difference'] < 300): 
    abandoned_during_MOH += 1 

print('Number of Calls Abandoned During Hold Music:', abandoned_during_MOH)

Number of Calls Abandoned During Hold Music: 1


In [13]:
def count_conditioned_events(music_calls, term_col='Termination Reason', term_val='Customer Left',
                             activity_col='Activity Name', activity_val='PlayMOH300s',
                             time_col='Time Difference', time_threshold=300):
    mask = (
        (music_calls[term_col] == term_val) &
        (music_calls[activity_col].shift(1) == activity_val) &
        (music_calls[time_col] < time_threshold)
    )
    return mask.sum()



abandoned_during_MOH = count_conditioned_events(music_calls)

print('Number of Calls Abandoned while Listening to Hold Music:', int(abandoned_during_MOH))


# 
music_calls['Abandoned During MOH'] = ((music_calls['Termination Reason'] == 'Abandoned') &  
                                       (music_calls['Activity Name'].shift(1) == 'PlayMOH300s') 
                                       &  (music_calls['Time Difference'] < 300))

music_calls

Number of Calls Abandoned while Listening to Hold Music: 343


,Call ID,Contact Session ID,EP Name,Flow Name,Activity Name,Queue Name,Agent Name,Termination Reason,Activity Start Timestamp,Time Difference,Duration,Abandoned During MOH
0,37,25390a18-15db-4e03-b5f2-bdc52d856154,Main Number Telephony EP,NaN,NaN,NaN,NaN,NaN,2025-03-17 07:59:51,0.0,0 days 00:25:34,False
1,37,25390a18-15db-4e03-b5f2-bdc52d856154,NaN,LACMain,NaN,NaN,NaN,NaN,2025-03-17 07:59:51,0.0,0 days 00:25:34,False
2,37,25390a18-15db-4e03-b5f2-bdc52d856154,Main Number Telephony EP,NaN,LanguageSelectionMenu,NaN,NaN,NaN,2025-03-17 07:59:51,0.0,0 days 00:25:34,False
3,37,25390a18-15db-4e03-b5f2-bdc52d856154,Main Number Telephony EP,LACMain,NaN,NaN,NaN,NaN,2025-03-17 07:59:51,0.0,0 days 00:25:34,False
4,37,25390a18-15db-4e03-b5f2-bdc52d856154,Main Number Telephony EP,NaN,MainMenu,NaN,NaN,NaN,2025-03-17 08:00:02,11.0,0 days 00:25:34,False
...,...,...,...,...,...,...,...,...,...,...,...,...
119564,68866,6b3acf29-e8e2-43ac-b434-ab6b5c888da3,All LAC Queues Telephony EP,NaN,NaN,NaN,NaN,NaN,2025-03-28 11:20:34,0.0,0 days 00:04:00,False
119565,68866,6b3acf29-e8e2-43ac-b434-ab6b5c888da3,All LAC Queues Telephony EP,NaN,NaN,Housing SubSeniors,NaN,NaN,2025-03-28 11:20:34,0.0,0 days 00:04:00,False
119566,68866,6b3acf29-e8e2-43ac-b434-ab6b5c888da3,All LAC Queues Telephony EP,NaN,PreQueueMessage2,NaN,NaN,NaN,2025-03-28 11:20:34,0.0,0 days 00:04:00,False
119567,68866,6b3acf29-e8e2-43ac-b434-ab6b5c888da3,All LAC Queues Telephony EP,NaN,PlayMOH300s,NaN,NaN,NaN,2025-03-28 11:20:46,12.0,0 days 00:04:00,False


### Identifying Calls with an Activity in Final Row 

In [14]:
termination_row = calls_customer_left.loc[calls_customer_left['Termination Reason'] == 'Customer Left']

termination_row.loc[~termination_row['Activity Name'].isna()]


,Call ID,Contact Session ID,EP Name,Flow Name,Activity Name,Queue Name,Agent Name,Termination Reason,Activity Start Timestamp,Time Difference,Duration


In [15]:
# Finding the call with the largest number of steps 
calls_customer_left['Call ID'].value_counts().idxmax()

np.int64(8173)

In [16]:
# Example of a Customer being confused by the phone menu & being unable to find where to go 
    # bounce between PreTenant & Housing Menu for 2 hours without entering a queue 

pd.set_option('display.max_rows', None)

calls_customer_left[calls_customer_left['Call ID'] == 8173]

,Call ID,Contact Session ID,EP Name,Flow Name,Activity Name,Queue Name,Agent Name,Termination Reason,Activity Start Timestamp,Time Difference,Duration
115050,8173,9da4f945-76fe-422c-bff5-1acf63c9d723,Main Number Telephony EP,NaN,NaN,NaN,NaN,NaN,2025-04-21 08:04:40,0.0,0 days 02:13:29
115051,8173,9da4f945-76fe-422c-bff5-1acf63c9d723,NaN,LACMain,NaN,NaN,NaN,NaN,2025-04-21 08:04:40,0.0,0 days 02:13:29
115052,8173,9da4f945-76fe-422c-bff5-1acf63c9d723,Main Number Telephony EP,NaN,LanguageSelectionMenu,NaN,NaN,NaN,2025-04-21 08:04:40,0.0,0 days 02:13:29
115053,8173,9da4f945-76fe-422c-bff5-1acf63c9d723,Main Number Telephony EP,LACMain,NaN,NaN,NaN,NaN,2025-04-21 08:04:40,0.0,0 days 02:13:29
115068,8173,9da4f945-76fe-422c-bff5-1acf63c9d723,Main Number Telephony EP,NaN,LanguageSelectionMenu,NaN,NaN,NaN,2025-04-21 08:04:58,18.0,0 days 02:13:29
115075,8173,9da4f945-76fe-422c-bff5-1acf63c9d723,Main Number Telephony EP,NaN,MainMenu,NaN,NaN,NaN,2025-04-21 08:05:08,10.0,0 days 02:13:29
115082,8173,9da4f945-76fe-422c-bff5-1acf63c9d723,NaN,PreLegalMenuSeniorsMenu,NaN,NaN,NaN,NaN,2025-04-21 08:05:31,23.0,0 days 02:13:29
115083,8173,9da4f945-76fe-422c-bff5-1acf63c9d723,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SeniorsMenu,NaN,NaN,NaN,2025-04-21 08:05:31,0.0,0 days 02:13:29
115091,8173,9da4f945-76fe-422c-bff5-1acf63c9d723,NaN,LegalMenu,NaN,NaN,NaN,NaN,2025-04-21 08:05:39,8.0,0 days 02:13:29
115092,8173,9da4f945-76fe-422c-bff5-1acf63c9d723,Legal Menu Telephony EP,NaN,LegalMenu1,NaN,NaN,NaN,2025-04-21 08:05:39,0.0,0 days 02:13:29
